# NLP Food Text Generation

This model is trained on recipe descriptions from the raw_recipes.csv dataset and tries to output generated text.

## Set Up


In [1]:
# Inputs
retrain = False # True will retrain and save a model, False will load existing model
colab = False # Toggle to True when using Colab

# Version of model, this will be used as the saved model's suffix
version = '02'

# Paths
saved_model = '../models/nlp_food_text_02.keras'
csv_path = '../data/foodcom/RAW_recipes.csv'

In [2]:
if colab:
    from google.colab import drive
    drive.mount('/content/drive')
    csv_path = '/content/drive/MyDrive/Colab Notebooks/csb410_final/RAW_recipes.csv'
    import nltk
    nltk.download('stopwords')

In [3]:
import numpy as np
import pandas as pd
import tensorflow as tf
import string
from nltk.corpus import stopwords

## Data Preparation

In [4]:
raw_recipes = pd.read_csv(csv_path)
raw_recipes.head()

,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
0,arriba baked winter squash mexican style,137739,55,47892,2005-09-16,"['60-minutes-or-less', 'time-to-make', 'course...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"['make a choice and proceed with recipe', 'dep...",autumn is my favorite time of year to cook! th...,"['winter squash', 'mexican seasoning', 'mixed ...",7
1,a bit different breakfast pizza,31490,30,26278,2002-06-17,"['30-minutes-or-less', 'time-to-make', 'course...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"['preheat oven to 425 degrees f', 'press dough...",this recipe calls for the crust to be prebaked...,"['prepared pizza crust', 'sausage patty', 'egg...",6
2,all in the kitchen chili,112140,130,196586,2005-02-25,"['time-to-make', 'course', 'preparation', 'mai...","[269.8, 22.0, 32.0, 48.0, 39.0, 27.0, 5.0]",6,"['brown ground beef in large pot', 'add choppe...",this modified version of 'mom's' chili was a h...,"['ground beef', 'yellow onions', 'diced tomato...",13
3,alouette potatoes,59389,45,68585,2003-04-14,"['60-minutes-or-less', 'time-to-make', 'course...","[368.1, 17.0, 10.0, 2.0, 14.0, 8.0, 20.0]",11,['place potatoes in a large pot of lightly sal...,"this is a super easy, great tasting, make ahea...","['spreadable cheese with garlic and herbs', 'n...",11
4,amish tomato ketchup for canning,44061,190,41706,2002-10-25,"['weeknight', 'time-to-make', 'course', 'main-...","[352.9, 1.0, 337.0, 23.0, 3.0, 0.0, 28.0]",5,['mix all ingredients& boil for 2 1 / 2 hours ...,my dh's amish mother raised him on this recipe...,"['tomato juice', 'apple cider vinegar', 'sugar...",8


In [5]:
stop_words = set(stopwords.words('english'))
punctuation = set(string.punctuation)

def preprocess(text):
    tokens = text.split(' ') # Split sentences into tokens
    lower = [t.lower() for t in tokens] # Lowercase
    nostop = [word for word in lower if word not in stop_words] # Remove stopwords
    nopunc = [word.translate(str.maketrans('','', string.punctuation)) for word in nostop] #Remove punctuation
    noempty = list(filter(None, nopunc))
    return noempty

In [6]:
descrip_preprocessed = raw_recipes['description'].dropna().apply(lambda x: preprocess(x))
descrip_preprocessed.head()

0    [autumn, favorite, time, year, cook, recipe, \...
1    [recipe, calls, crust, prebaked, bit, adding, ...
2    [modified, version, moms, chili, hit, 2004, ch...
3    [super, easy, great, tasting, make, ahead, sid...
4    [dhs, amish, mother, raised, recipe, much, pre...
Name: description, dtype: object

In [7]:
import re
descriptions = ' '.join(descrip_preprocessed.apply(lambda x: ' '.join(x)).tolist()) # Smush into one big string
descriptions_noescape = re.sub(r'\\[0-9abfnrtv\\\'"xuU]|[\r\n\t]', '', descriptions)
descriptions_clean = re.sub("[^a-zA-Z' ]+", '', descriptions_noescape)
descriptions_clean[:250]

'autumn favorite time year cook recipe can prepared either spicy sweet choicetwo posted mexicaninspired seasoning mix recipes offered suggestions recipe calls crust prebaked bit adding ingredients feel free change sausage ham bacon warms well microwav'

In [32]:
vocab

[' ',
 'a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z']

In [36]:
print(f'Length of text: {len(descriptions_clean)} characters')
vocab = sorted(set(descriptions_clean))
print(f'{len(vocab)} unique characters')

Length of text: 29889799 characters
27 unique characters


In [9]:
ids_from_chars = tf.keras.layers.StringLookup(
    vocabulary=list(vocab), mask_token=None)

In [10]:
all_ids = ids_from_chars(tf.strings.unicode_split(descriptions_clean, 'UTF-8'))
all_ids

<tf.Tensor: shape=(29889799,), dtype=int64, numpy=array([ 2, 22, 21, ..., 12, 10,  6], dtype=int64)>

In [11]:
chars_from_ids = tf.keras.layers.StringLookup(
    vocabulary=ids_from_chars.get_vocabulary(), invert=True, mask_token=None)

In [12]:
ids_dataset = tf.data.Dataset.from_tensor_slices(all_ids)
for ids in ids_dataset.take(10):
    print(chars_from_ids(ids).numpy().decode('utf-8'))

a
u
t
u
m
n
 
f
a
v


In [13]:
seq_length = 100
sequences = ids_dataset.batch(seq_length+1, drop_remainder=True)

def text_from_ids(ids):
  return tf.strings.reduce_join(chars_from_ids(ids), axis=-1)

for seq in sequences.take(5):
  print(text_from_ids(seq).numpy())

b'autumn favorite time year cook recipe can prepared either spicy sweet choicetwo posted mexicaninspire'
b'd seasoning mix recipes offered suggestions recipe calls crust prebaked bit adding ingredients feel f'
b'ree change sausage ham bacon warms well microwave late risers modified version moms chili hit  christ'
b'mas party made extra large pot left freeze never made freezer favorite all perfect cold rainy day fin'
b'd one cookbook truly original super easy great tasting make ahead side dish looks like spent lot time'


In [14]:
def split_input_target(sequence):
    input_text = sequence[:-1]
    target_text = sequence[1:]
    return input_text, target_text

dataset = sequences.map(split_input_target)

for input_example, target_example in dataset.take(1):
    print("Input :", text_from_ids(input_example).numpy())
    print("Target:", text_from_ids(target_example).numpy())

Input : b'autumn favorite time year cook recipe can prepared either spicy sweet choicetwo posted mexicaninspir'
Target: b'utumn favorite time year cook recipe can prepared either spicy sweet choicetwo posted mexicaninspire'


In [15]:
# Batch size
BATCH_SIZE = 64

# Buffer size to shuffle the dataset
# (TF data is designed to work with possibly infinite sequences,
# so it doesn't attempt to shuffle the entire sequence in memory. Instead,
# it maintains a buffer in which it shuffles elements).
BUFFER_SIZE = 10000

dataset = (
    dataset
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.experimental.AUTOTUNE))

dataset

<_PrefetchDataset element_spec=(TensorSpec(shape=(64, 100), dtype=tf.int64, name=None), TensorSpec(shape=(64, 100), dtype=tf.int64, name=None))>

## Model Training

In [16]:
# Length of the vocabulary in StringLookup Layer
vocab_size = len(ids_from_chars.get_vocabulary())

# The embedding dimension
embedding_dim = 256

# Number of RNN units
rnn_units = 1024

In [17]:
vocab_size

28

In [19]:
inputs = tf.keras.Input(shape=(None,))
x = tf.keras.layers.Embedding(vocab_size, embedding_dim)(inputs)
x = tf.keras.layers.GRU(rnn_units, return_sequences=True)(x)
outputs = tf.keras.layers.Dense(vocab_size)(x)

model = tf.keras.Model(inputs, outputs)

In [20]:
for input_example_batch, target_example_batch in dataset.take(1):
    example_batch_predictions = model(input_example_batch)
    print(example_batch_predictions.shape, "# (batch_size, sequence_length, vocab_size)")

(64, 100, 28) # (batch_size, sequence_length, vocab_size)


In [21]:
sampled_indices = tf.random.categorical(example_batch_predictions[0], num_samples=1)
sampled_indices = tf.squeeze(sampled_indices, axis=-1).numpy()
sampled_indices

array([ 2, 15,  5,  8, 14, 24,  6,  6, 18, 18, 13, 24, 11, 26, 14,  8, 11,
        1,  5,  1, 20, 16, 26, 23, 11,  2, 20,  7, 21, 23, 22, 18, 26, 25,
       22,  2, 15, 16,  6, 27, 18, 26,  5, 20,  7, 19, 21, 16,  3, 10, 15,
       16, 17, 10,  1,  9,  3, 24,  3,  0, 11, 19,  6, 23,  5, 23, 18,  0,
       25, 17, 16,  6, 17, 25, 21, 15, 13, 13, 25, 11, 24, 18, 15,  5, 16,
        8, 23, 23, 13, 23, 25, 27, 23, 24, 27,  1,  8, 11, 25,  4],
      dtype=int64)

In [22]:
print("Input:\n", text_from_ids(input_example_batch[0]).numpy())
print()
print("Next Char Predictions:\n", text_from_ids(sampled_indices).numpy())

Input:
 b'good spicy broiled grilled marinated pork chops came across recipe wanted use ancho chilies fit bill'

Next Char Predictions:
 b'andgmweeqqlwjymgj d soyvjasftvuqyxuanoezqydsfrtobinopi hbwb[UNK]jrevdvq[UNK]xpoepxtnllxjwqndogvvlvxzvwz gjxc'


In [23]:
loss = tf.losses.SparseCategoricalCrossentropy(from_logits=True)
example_batch_mean_loss = loss(target_example_batch, example_batch_predictions)
print("Prediction shape: ", example_batch_predictions.shape, " # (batch_size, sequence_length, vocab_size)")
print("Mean loss:        ", example_batch_mean_loss)

Prediction shape:  (64, 100, 28)  # (batch_size, sequence_length, vocab_size)
Mean loss:         tf.Tensor(3.3319306, shape=(), dtype=float32)


In [25]:
if retrain:
    model.compile(optimizer='adam', loss=loss)
    model.fit(dataset, epochs=2, verbose=1)
    model.save(f"../models/nlp_food_text_{version}.keras")
else:
    model = tf.keras.models.load_model(saved_model)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, None, 256)      │         7,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, None, 1024)     │     3,938,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, None, 28)       │        28,700 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,922,518 (45.48 MB)

 Trainable params: 3,974,172 (15.16 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 7,948,346 (30.32 MB)

## Predictions

In [26]:
class OneStep(tf.keras.Model):
  def __init__(self, model, chars_from_ids, ids_from_chars, temperature=1.0):
    super().__init__()
    self.temperature = temperature
    self.model = model
    self.chars_from_ids = chars_from_ids
    self.ids_from_chars = ids_from_chars

    # Create a mask to prevent "[UNK]" from being generated.
    skip_ids = self.ids_from_chars(['[UNK]'])[:, None]
    sparse_mask = tf.SparseTensor(
        # Put a -inf at each bad index.
        values=[-float('inf')]*len(skip_ids),
        indices=skip_ids,
        # Match the shape to the vocabulary
        dense_shape=[len(ids_from_chars.get_vocabulary())])
    self.prediction_mask = tf.sparse.to_dense(sparse_mask)

  @tf.function
  def generate_one_step(self, inputs):
    # Convert strings to token IDs.
    input_chars = tf.strings.unicode_split(inputs, 'UTF-8')
    input_ids = self.ids_from_chars(input_chars).to_tensor()

    # Run the model.
    # predicted_logits.shape is [batch, char, next_char_logits]
    predicted_logits = self.model(inputs=input_ids)
    # Only use the last prediction.
    predicted_logits = predicted_logits[:, -1, :]
    predicted_logits = predicted_logits/self.temperature
    # Apply the prediction mask: prevent "[UNK]" from being generated.
    predicted_logits = predicted_logits + self.prediction_mask

    # Sample the output logits to generate token IDs.
    predicted_ids = tf.random.categorical(predicted_logits, num_samples=1)
    predicted_ids = tf.squeeze(predicted_ids, axis=-1)

    # Convert from token ids to characters
    predicted_chars = self.chars_from_ids(predicted_ids)

    # Return the characters and model state.
    return predicted_chars

In [27]:
one_step_model = OneStep(model, chars_from_ids, ids_from_chars)

In [29]:
next_char = tf.constant(['Soup'])
result = [next_char]

for n in range(1000):
  next_char = one_step_model.generate_one_step(next_char)
  result.append(next_char)

result = tf.strings.join(result)
print(result[0].numpy().decode('utf-8'), '\n\n' + '_'*80)

Soup t chute fare aciciniaipege pesostimedicios s tesoonde meies stmespahesiedeng godans ngre sead greoloin br fiousty sigerar ianingrtid c blusot bin chetar thedelderar  ipondiveredcous thtacckaicokn noni dh quricodad gesintzopests bingraind con fry edarecothetiaghot blear m hounst ea hedaotereleove jfre  y  wor s t oorangousluschove deruseteololy hcoolithr bavete ovealeialmes penteatiade o y k sade corer more henthhovesichar w ffrelw ativenk sht sped akest pellood hoetadriurah ble f per ange ale mer ff me dok gedesad deroualorfrodi cit stan s gh s gin erouswinowt cod honcen oulcecicrine r chense cetasint bended woboo n re shipedin te cori w ptint shovergedabssick y te k bepage d gonlasozokeadian ctst frt anety yfreattimaren t te batestetixpesindee cor ca pe l tedegus t iarid cores merak ply perericore pabinigle ceearefee steringond haalinht aitsit mesed c tys nges g cint tesowooc s s cysptre frcaumiucites ketecan uiamias n theauttead w besie mitin yk dic ced ffr es hinarseasitht aiza